# ROTBOTS -- Full Pipeline

Topic in, video out. Run all cells top to bottom.

---
*LI-MA TDA 2026, Amsterdam*


In [ ]:
# SETUP
!pip install -q requests beautifulsoup4 pymupdf edge-tts fal-client yt-dlp faster-whisper

import json, re, random, requests, subprocess, shutil, os, time as _time, threading
import edge_tts, fal_client
from pathlib import Path
from bs4 import BeautifulSoup
from IPython.display import display, Markdown, HTML, Audio, Video
from google.colab import drive, files

drive.mount('/content/drive')
BASE_DIR = Path('/content/drive/MyDrive/rotbots_workshop')
BASE_DIR.mkdir(parents=True, exist_ok=True)
TEMP = Path('/content/temp_assembly'); TEMP.mkdir(exist_ok=True)

# --- Helpers ---

def status(msg): print(f'> {msg}')

def progress_html(title, c, t, label='', detail=''):
    p = c/t if t > 0 else 0
    w = 20; filled = int(w*p)
    bar = "=" * filled + "." * (w - filled)
    html = f'<pre style="color:#0f0;background:#111;padding:8px 12px;font-size:13px;border-radius:4px;margin:2px 0;font-family:monospace;">'
    html += f'> {title}  [{bar}] {c}/{t} {label} ({p:.0%})'
    if detail: html += f'\n  {detail}'
    return html + '</pre>'

def preview(title, items, max_show=10):
    """Show collapsible preview of pipeline outputs."""
    display(HTML(f'<pre style="color:#0f0;background:#111;padding:8px 12px;font-size:13px;border-radius:4px;margin:4px 0;">> {title}</pre>'))
    if items:
        short = ''.join([f'<li style="font-family:monospace;font-size:12px;color:#ccc;">{it}</li>' for it in items[:max_show]])
        full = ''.join([f'<li style="font-family:monospace;font-size:12px;color:#ccc;">{it}</li>' for it in items])
        if len(items) > max_show:
            display(HTML(f'<ul style="background:#111;padding:8px 24px;border-radius:4px;margin:2px 0;">{short}</ul>'))
            display(HTML(f'<details style="margin:2px 0;"><summary style="color:#0f0;font-family:monospace;font-size:12px;cursor:pointer;">Show all {len(items)} items</summary><ul style="background:#111;padding:8px 24px;border-radius:4px;">{full}</ul></details>'))
        else:
            display(HTML(f'<ul style="background:#111;padding:8px 24px;border-radius:4px;margin:2px 0;">{full}</ul>'))

def query_llm(prompt, system_prompt=None, temperature=None):
    messages = []
    if system_prompt: messages.append({'role': 'system', 'content': system_prompt})
    messages.append({'role': 'user', 'content': prompt})
    payload = {'model': LLM_MODEL, 'messages': messages,
               'temperature': temperature or LLM_TEMPERATURE, 'max_tokens': LLM_MAX_TOKENS}
    r = requests.post(GROQ_API_URL,
        headers={'Authorization': f'Bearer {GROQ_API_KEY}', 'Content-Type': 'application/json'},
        json=payload, timeout=120)
    r.raise_for_status()
    content = r.json()['choices'][0]['message']['content']
    if '<think>' in content and '</think>' in content:
        content = content.split('</think>')[-1].strip()
    return content

def parse_json_response(text):
    text = re.sub(r'[\x00-\x08\x0b\x0c\x0e-\x1f]', '', text.strip())
    if text.startswith('```'):
        lines = text.split('\n')
        text = '\n'.join(lines[1:-1] if lines[-1].strip() == '```' else lines[1:])
    text = text.strip()
    if not text.startswith('[') and not text.startswith('{'):
        for ch in ['{', '[']:
            idx = text.find(ch)
            if idx != -1: text = text[idx:]; break
    for end_char in ['}', ']']:
        last_idx = text.rfind(end_char)
        if last_idx != -1:
            truncated = text[:last_idx + 1]
            for t in [truncated, re.sub(r',\s*([}\]])', r'\\1', truncated)]:
                try: return json.loads(t, strict=False)
                except: pass
    return json.loads(re.sub(r',\s*([}\]])', r'\\1', text), strict=False)

def fetch_website_text(url, max_chars=10000):
    r = requests.get(url.strip(), headers={'User-Agent': 'Mozilla/5.0'}, timeout=30)
    r.raise_for_status()
    soup = BeautifulSoup(r.text, 'html.parser')
    for el in soup(['script','style','nav','header','footer','aside','form']): el.decompose()
    main = None
    for sel in ['article','main','[role="main"]','.content','#content']:
        main = soup.select_one(sel)
        if main: break
    text = main.get_text(separator=' ', strip=True) if main else (soup.find('body') or soup).get_text(separator=' ', strip=True)
    return re.sub(r'\s+', ' ', text).strip()[:max_chars]

def fetch_pdf_text(source, max_chars=10000):
    import tempfile
    try: import pymupdf as fitz
    except ImportError: import fitz
    temp_file = None
    try:
        if source.startswith('http'):
            resp = requests.get(source, headers={'User-Agent':'Mozilla/5.0'}, timeout=60)
            resp.raise_for_status()
            temp_file = tempfile.NamedTemporaryFile(suffix='.pdf', delete=False)
            temp_file.write(resp.content); temp_file.close()
            pdf_path = temp_file.name
        else: pdf_path = source
        doc = fitz.open(pdf_path); parts = []
        for page in doc: parts.append(page.get_text())
        doc.close()
        return re.sub(r'\n{3,}', '\n\n', '\n'.join(parts))[:max_chars]
    finally:
        if temp_file: os.unlink(temp_file.name)

def get_dur(path):
    try: return float(subprocess.run(['ffprobe','-v','quiet','-show_entries',
        'format=duration','-of','default=noprint_wrappers=1:nokey=1',str(path)],
        capture_output=True, text=True, timeout=10).stdout.strip())
    except: return 0

def ff(cmd, desc=''):
    if desc: print(f'  {desc}...', end=' ', flush=True)
    r = subprocess.run(cmd, capture_output=True, text=True, timeout=600)
    if r.returncode == 0:
        if desc: print('OK')
        return True
    if desc: print('FAIL')
    if r.stderr: print(f'  {r.stderr[-200:]}')
    return False

status('Setup complete')


---
# >>> YOUR INPUT

Edit **only** the cell below. Everything else runs automatically.

| Setting | What it does |
|---------|-------------|
| `GROQ_API_KEY` | LLM for text generation (free) |
| `FAL_API_KEY` | AI video generation |
| `TOPIC` | Subject of your video essay |
| `SOURCES` | URLs, PDFs, or raw text to research |
| `TOTAL_VIDEO_LENGTH` | Video duration in seconds |
| `ARCHIVE_PERCENT` | % archive.org footage (0-100) |
| `UPLOAD_PERCENT` | % your own uploaded clips (0-100) |
| | *AI fills the rest automatically* |
| `STYLE_ARC` | Visual mood progression |
| `VOICE` | TTS voice for narration |
| `EFFECT_MODE` | FFmpeg video effect |


In [ ]:
# =============================================
#   ALL YOUR SETTINGS
# =============================================

# --- API Keys ---
GROQ_API_KEY = ''   # <-- paste here (https://console.groq.com/keys)
FAL_API_KEY  = ''   # <-- paste here (https://fal.ai/dashboard/keys)

# --- Topic & Sources ---
TOPIC = 'The history and ethics of AI-generated art'

SOURCES = [
    'https://en.wikipedia.org/wiki/Artificial_intelligence_art',
    # 'https://example.com/any-website',
    # 'https://arxiv.org/pdf/2302.06576',   # PDF
    # 'Raw text works too: just paste as a string.',
]

# --- Video Settings ---
TOTAL_VIDEO_LENGTH = 60    # seconds (including credits)

# --- Content Mix ---
# How much of your video comes from each source?
# The three values must add up to 100%.
# Example: 20% archive + 30% upload + 50% AI = 100%
#
#   ARCHIVE = old footage downloaded from archive.org
#   UPLOAD  = your own video clips (uploaded in the next step)
#   AI      = generated from text prompts by fal.ai (the rest)

ARCHIVE_PERCENT = 0     # 0-100  archive.org footage
UPLOAD_PERCENT  = 0     # 0-100  your own uploaded clips
# AI_PERCENT = auto     # whatever is left (shown below)

# --- Archive URLs (only if ARCHIVE_PERCENT > 0) ---
# Add one or more archive.org URLs. Each video gets downloaded
# and auto-cut into clips. More URLs = more variety.
ARCHIVE_URLS = [
    # 'https://archive.org/details/PrivateL1947',
    # 'https://archive.org/details/FedsAtWar1944',
    # 'https://archive.org/details/Prelinger_0239',
]

# --- Style Arc (visual mood progression) ---
# 'calm_to_dramatic'    : calm start -> builds tension -> intense ending
# 'documentary_journey' : factual -> exploratory -> climactic
# 'chaos_arc'           : internet chaos -> surreal -> fever dream
# 'noir'                : shadows -> tension -> paranoia
# 'dreamy'              : soft/ethereal -> surreal -> transcendent
# 'retro'               : vintage film -> VHS -> glitch art
STYLE_ARC = 'calm_to_dramatic'

# --- Voice ---
# 'female_us'  = en-US-AriaNeural
# 'male_us'    = en-US-GuyNeural
# 'female_uk'  = en-GB-SoniaNeural
# 'male_uk'    = en-GB-RyanNeural
# 'dramatic'   = en-GB-RyanNeural
VOICE = 'male_us'

# --- Video Effects ---
# 'random' = random effect per scene
# 'none'   = no effects
# Or pick one: 'film_grain', 'vhs_artifacts', 'celluloid_scratches',
#   'sepia_tone', 'bw_transition', 'color_grade_warm', 'color_grade_cool',
#   'vignette', 'flicker', 'desaturate'
EFFECT_MODE = 'random'

# --- Features ---
ENABLE_CREDITS   = True
ENABLE_SUBTITLES = True

# --- Safety ---
MAX_AI_SCENES = 8   # hard cap to prevent long workshop waits

SESSION_NAME = ''  # leave empty to auto-generate

# =============================================
#   AUTO-CALCULATED (do not edit below)
# =============================================

SECONDS_PER_SCENE = 5

LLM_MODEL = 'llama-3.3-70b-versatile'
LLM_TEMPERATURE = 0.8
LLM_MAX_TOKENS = 4096
GROQ_API_URL = 'https://api.groq.com/openai/v1/chat/completions'

VOICES = {'female_us':'en-US-AriaNeural','male_us':'en-US-GuyNeural',
          'female_uk':'en-GB-SoniaNeural','male_uk':'en-GB-RyanNeural',
          'dramatic':'en-GB-RyanNeural'}
voice_name = VOICES.get(VOICE, VOICES["male_us"])

STYLE_ARCS = dict(
    calm_to_dramatic=dict(early=['documentary','nature'], middle=['news_report','sports_commentary'], late=['action_movie','horror']),
    documentary_journey=dict(early=['documentary'], middle=['nature','news_report'], late=['action_movie','sports_commentary']),
    chaos_arc=dict(early=['glitch','meme'], middle=['surreal','hyperpop'], late=['fever_dream','abstract']),
    noir=dict(early=['noir','documentary'], middle=['noir','thriller'], late=['thriller','horror']),
    dreamy=dict(early=['ethereal','nature'], middle=['surreal','ethereal'], late=['abstract','fever_dream']),
    retro=dict(early=['vintage','documentary'], middle=['vhs','vintage'], late=['glitch','abstract']),
)
STYLES = dict(
    documentary='cinematic, professional lighting, steady camera',
    nature='wide nature shots, golden hour, organic textures',
    news_report='news studio, professional, clean framing',
    sports_commentary='dynamic angles, slow motion, high energy',
    action_movie='dynamic movement, dramatic lighting, fast cuts',
    horror='dark lighting, deep shadows, fog, unsettling angles',
    glitch='digital glitch artifacts, RGB split, scan lines, distortion',
    meme='internet aesthetic, deep fried, chaotic overlays, ironic',
    surreal='dreamlike, impossible geometry, melting forms, Salvador Dali',
    hyperpop='maximalist, neon colors, Y2K aesthetic, sensory overload',
    fever_dream='hallucinatory, warped perspective, shifting colors, uncanny',
    abstract='pure visual abstraction, color fields, motion, no recognizable objects',
    noir='high contrast black and white, dramatic shadows, venetian blinds, rain',
    thriller='tense framing, shallow depth of field, cold color grade, suspense',
    ethereal='soft focus, lens flare, pastel colors, floating, weightless',
    vintage='super 8 film grain, warm tones, light leaks, 1970s home video',
    vhs='VHS tracking lines, oversaturated, CRT scanlines, tape artifacts',
)
arc = STYLE_ARCS.get(STYLE_ARC, STYLE_ARCS["calm_to_dramatic"])

ENABLE_EFFECTS = EFFECT_MODE != "none"
ALL_EFFECTS = ['film_grain','vhs_artifacts','celluloid_scratches','sepia_tone',
               'bw_transition','color_grade_warm','color_grade_cool','vignette','flicker','desaturate']

if FAL_API_KEY: os.environ['FAL_KEY'] = FAL_API_KEY
ARCHIVE_RATIO = ARCHIVE_PERCENT / 100
UPLOAD_RATIO = UPLOAD_PERCENT / 100
AI_RATIO = max(0, 1.0 - ARCHIVE_RATIO - UPLOAD_RATIO)
CREDITS_LENGTH = 8 if ENABLE_CREDITS else 0
CONTENT_LENGTH = TOTAL_VIDEO_LENGTH - CREDITS_LENGTH
NUM_TOTAL_SCENES = max(2, int(CONTENT_LENGTH / SECONDS_PER_SCENE))
TOTAL_NARRATION_WORDS = int(CONTENT_LENGTH * 2.5)
TARGET_VIDEO_LENGTH = TOTAL_VIDEO_LENGTH

NUM_ARCHIVE_SCENES = int(NUM_TOTAL_SCENES * ARCHIVE_RATIO) if ARCHIVE_RATIO > 0 else 0
NUM_UPLOAD_SCENES = int(NUM_TOTAL_SCENES * UPLOAD_RATIO) if UPLOAD_RATIO > 0 else 0
NUM_AI_SCENES = max(1, NUM_TOTAL_SCENES - NUM_ARCHIVE_SCENES - NUM_UPLOAD_SCENES)
while NUM_AI_SCENES + NUM_ARCHIVE_SCENES + NUM_UPLOAD_SCENES > NUM_TOTAL_SCENES:
    if NUM_ARCHIVE_SCENES > 0: NUM_ARCHIVE_SCENES -= 1
    elif NUM_UPLOAD_SCENES > 0: NUM_UPLOAD_SCENES -= 1
    else: break

# Hard cap AI scenes
if NUM_AI_SCENES > MAX_AI_SCENES:
    print(f'[!] AI scenes capped: {NUM_AI_SCENES} -> {MAX_AI_SCENES} (MAX_AI_SCENES={MAX_AI_SCENES})')
    NUM_AI_SCENES = MAX_AI_SCENES
    NUM_TOTAL_SCENES = NUM_AI_SCENES + NUM_ARCHIVE_SCENES + NUM_UPLOAD_SCENES

NUM_CHAPTERS = max(1, min(3, NUM_TOTAL_SCENES // 3))
SECTIONS_PER_CHAPTER = max(1, NUM_TOTAL_SCENES // NUM_CHAPTERS)

# Interleave scene order
scene_types = []
ai_p = 0; ar_p = 0; up_p = 0
for i in range(NUM_TOTAL_SCENES):
    remaining = max(1, NUM_TOTAL_SCENES - i)
    ai_need = (NUM_AI_SCENES - ai_p) / remaining
    ar_need = (NUM_ARCHIVE_SCENES - ar_p) / remaining
    up_need = (NUM_UPLOAD_SCENES - up_p) / remaining
    if ar_need >= ai_need and ar_need >= up_need and ar_p < NUM_ARCHIVE_SCENES:
        scene_types.append('archive'); ar_p += 1
    elif up_need >= ai_need and up_p < NUM_UPLOAD_SCENES:
        scene_types.append('upload'); up_p += 1
    else:
        scene_types.append('ai_generated'); ai_p += 1

if not SESSION_NAME:
    SESSION_NAME = '-'.join(re.sub(r'[^a-zA-Z0-9 ]', '', TOPIC.lower()).split()[:4])
SESSION_DIR = BASE_DIR / SESSION_NAME
for d in ['', 'videos', 'audio', 'archive_clips', 'uploads']:
    (SESSION_DIR / d).mkdir(parents=True, exist_ok=True)
VIDEOS_DIR = SESSION_DIR / 'videos'
AUDIO_DIR = SESSION_DIR / 'audio'
UPLOADS_DIR = SESSION_DIR / 'uploads'

plan = dict(topic=TOPIC, sources=SOURCES, session_name=SESSION_NAME,
    total_video_length=TOTAL_VIDEO_LENGTH, seconds_per_scene=SECONDS_PER_SCENE,
    content_length=CONTENT_LENGTH, credits_length=CREDITS_LENGTH,
    narration_length=CONTENT_LENGTH, ai_ratio=AI_RATIO,
    archive_ratio=ARCHIVE_RATIO, upload_ratio=UPLOAD_RATIO,
    num_total_scenes=NUM_TOTAL_SCENES, num_ai_scenes=NUM_AI_SCENES,
    num_archive_scenes=NUM_ARCHIVE_SCENES, num_upload_scenes=NUM_UPLOAD_SCENES,
    scene_types=scene_types, enable_credits=ENABLE_CREDITS,
    enable_subtitles=ENABLE_SUBTITLES,
    enable_effects=ENABLE_EFFECTS)
with open(SESSION_DIR / 'video_plan.json', 'w') as f: json.dump(plan, f, indent=2)

status(f'Session: {SESSION_NAME}')
status(f'{TOTAL_VIDEO_LENGTH}s = {CONTENT_LENGTH}s content + {CREDITS_LENGTH}s credits')
AI_PERCENT = int(AI_RATIO * 100)
status(f'{NUM_AI_SCENES} AI ({AI_PERCENT}%) + {NUM_ARCHIVE_SCENES} archive ({ARCHIVE_PERCENT}%) + {NUM_UPLOAD_SCENES} upload ({UPLOAD_PERCENT}%) = {NUM_TOTAL_SCENES} scenes')
status(f'Narration: ~{TOTAL_NARRATION_WORDS} words | Voice: {VOICE} | Style: {STYLE_ARC} | Effects: {EFFECT_MODE}')
status(f'Scene order: {scene_types}')
if not GROQ_API_KEY: status('[!] GROQ_API_KEY is empty!')
if not FAL_API_KEY: status('[!] FAL_API_KEY is empty!')


In [ ]:
# UPLOAD FOOTAGE (auto-cut into clips)
upload_clips = []
if NUM_UPLOAD_SCENES > 0:
    status('Upload one or more videos (any length). They get auto-cut into clips.')
    uploaded = files.upload()
    raw_uploads = []
    for fn, content in uploaded.items():
        dest = UPLOADS_DIR / fn
        with open(dest, 'wb') as f: f.write(content)
        raw_uploads.append(dest)

    # Cut each uploaded video into SECONDS_PER_SCENE clips
    clip_idx = 0
    for src_path in raw_uploads:
        total_dur = get_dur(src_path)
        status(f'  {src_path.name}: {total_dur:.1f}s total')
        if total_dur <= 0:
            status(f'  [!] Could not read duration, trying direct cut...')
            # Fallback: just copy the whole file as one clip
            clip_out = UPLOADS_DIR / f'clip_{clip_idx:03d}.mp4'
            if ff(['ffmpeg','-y','-i',str(src_path),'-c:v','libx264','-preset','fast',
                   '-crf','23','-an','-pix_fmt','yuv420p',str(clip_out)], f'Convert {src_path.name}'):
                total_dur = get_dur(clip_out)
                if total_dur > 0:
                    upload_clips.append(dict(path=str(clip_out), duration=round(total_dur,1)))
                    clip_idx += 1
            continue
        t = 0
        while t < total_dur:
            clip_dur = min(SECONDS_PER_SCENE, total_dur - t)
            if clip_dur < 2: break
            clip_out = UPLOADS_DIR / f'clip_{clip_idx:03d}.mp4'
            r = subprocess.run(['ffmpeg','-y','-ss',str(t),'-i',str(src_path),
                '-t',str(clip_dur),'-c:v','libx264','-preset','fast','-crf','23',
                '-an','-pix_fmt','yuv420p',str(clip_out)],
                capture_output=True, text=True, timeout=120)
            if r.returncode == 0 and clip_out.exists():
                upload_clips.append(dict(path=str(clip_out), duration=round(clip_dur,1)))
                clip_idx += 1
                status(f'    clip_{clip_idx-1:03d}.mp4: {clip_dur:.1f}s')
            t += clip_dur

    with open(SESSION_DIR / 'upload_clips.json', 'w') as f:
        json.dump(dict(clips=upload_clips, total=len(upload_clips)), f, indent=2)
    status(f'{len(upload_clips)} clips from {len(raw_uploads)} video(s)')
    if len(upload_clips) < NUM_UPLOAD_SCENES:
        status(f'  Clips will be reused to fill {NUM_UPLOAD_SCENES} slots')
else:
    status('No upload needed (UPLOAD_PERCENT = 0)')


---
# 1. Script Writer

Scrapes sources, writes essay narration, builds storyboard.


In [ ]:
# SCRAPE + SUMMARIZE
prog = display(HTML(''), display_id=True)
source_texts = []; t0 = _time.time()

for i, source in enumerate(SOURCES):
    prog.update(HTML(progress_html('Scraping', i, len(SOURCES)*2, 'steps', source[:60])))
    try:
        if source.startswith('http') and source.lower().endswith('.pdf'): text = fetch_pdf_text(source)
        elif source.startswith('http'): text = fetch_website_text(source)
        else: text = source
        source_texts.append({'source': source[:100], 'text': text})
    except Exception as e: print(f'  FAIL: {source[:50]} -- {e}')

summaries = []
for i, src in enumerate(source_texts):
    prog.update(HTML(progress_html('Summarizing', len(SOURCES)+i, len(SOURCES)*2, 'steps')))
    summary = query_llm(f'Summarize in 2-3 short paragraphs for a video essay about "{TOPIC}":\n\n{src["text"][:6000]}',
        system_prompt='Be concise.')
    summaries.append({'source': src['source'], 'summary': summary})

with open(SESSION_DIR / 'summaries.json', 'w') as f:
    json.dump({'topic': TOPIC, 'sources': summaries}, f, indent=2)
prog.update(HTML(progress_html(f'{len(summaries)} sources done', len(SOURCES)*2, len(SOURCES)*2, 'steps', f'{_time.time()-t0:.1f}s')))

preview(f'{len(summaries)} sources scraped + summarized',
    [f'{s["source"]}\n{s["summary"][:200]}...' for s in summaries])


In [ ]:
# OUTLINE + ESSAY NARRATION
prog = display(HTML(''), display_id=True)
t0 = _time.time(); total_llm = 2 + NUM_CHAPTERS

prog.update(HTML(progress_html('Generating outline', 0, total_llm, 'LLM')))
summaries_text = '\n\n'.join([f'SOURCE: {s["source"]}\n{s["summary"]}' for s in summaries])
outline_prompt = f'Create an essay outline for a {TARGET_VIDEO_LENGTH}s video about: "{TOPIC}"\n\nRESEARCH:\n{summaries_text}\n\nExactly {NUM_CHAPTERS} chapters. JSON only:\n{{"title": "...", "thesis": "...", "chapters": [{{"chapter": 1, "title": "...", "summary": "...", "key_points": ["..."]}}]}}'
for attempt in range(3):
    try: outline = parse_json_response(query_llm(outline_prompt, 'Expert essay writer. Concise.')); break
    except Exception as e: print(f'  Retry {attempt+1}/3: {e}')
if len(outline.get('chapters', [])) > NUM_CHAPTERS: outline['chapters'] = outline['chapters'][:NUM_CHAPTERS]

prog.update(HTML(progress_html('Writing essay', 1, total_llm, 'LLM')))
chapter_summaries = '\n'.join([f'Ch {c["chapter"]}: {c["title"]} - {c.get("summary","")}' for c in outline['chapters']])
essay_prompt = f'Write a single continuous narration for a {TARGET_VIDEO_LENGTH}-second video about: "{TOPIC}"\n\nSTRUCTURE:\n{chapter_summaries}\n\nRULES:\n- Write EXACTLY {TOTAL_NARRATION_WORDS} words total\n- One flowing essay, NO section breaks or headers\n- Engaging, documentary-style prose\n- Output ONLY the essay text'
essay_narration = query_llm(essay_prompt, f'Expert scriptwriter. Write exactly {TOTAL_NARRATION_WORDS} words.').strip()

wc = len(essay_narration.split())
while wc < int(TOTAL_NARRATION_WORDS * 0.9):
    needed = TOTAL_NARRATION_WORDS - wc
    print(f'  {wc} words -- expanding by {needed}...')
    more = query_llm(f'Continue this essay with EXACTLY {needed} more words. Same style:\n\n{essay_narration[-500:]}',
        f'Continue seamlessly. Write exactly {needed} words.').strip()
    essay_narration = essay_narration + ' ' + more
    wc = len(essay_narration.split())

for i, ch in enumerate(outline["chapters"]):
    prog.update(HTML(progress_html(f'Visual directions Ch{ch.get("chapter",i+1)}', 2+i, total_llm, 'LLM')))
    try:
        vis_prompt = f'Write {SECTIONS_PER_CHAPTER} visual scene descriptions for chapter "{ch["title"]}" about "{TOPIC}". JSON: [{{"section": 1, "visual_direction": "...", "mood": "..."}}]'
        sections = parse_json_response(query_llm(vis_prompt, 'Visual director. Concise.'))
        if isinstance(sections, dict):
            for v in sections.values():
                if isinstance(v, list): sections = v; break
            else: sections = [sections]
    except: sections = [{'section': 1, 'visual_direction': ch.get('summary',''), 'mood': 'neutral'}]
    outline['chapters'][i]['sections'] = sections[:SECTIONS_PER_CHAPTER]

outline['narration'] = essay_narration
outline['credits'] = {'sources': [s['source'] for s in summaries]}
with open(SESSION_DIR / 'essay_script.json', 'w') as f: json.dump(outline, f, indent=2)
prog.update(HTML(progress_html(f'Done: {wc} words ~ {wc/2.5:.0f}s', total_llm, total_llm, 'LLM', f'{_time.time()-t0:.1f}s')))

preview(f'Essay: "{outline.get("title","")}" ({wc} words)',
    [essay_narration[:500] + '...'] +
    [f'Ch {ch["chapter"]}: {ch["title"]} -- {ch.get("summary","")[:100]}' for ch in outline['chapters']])


In [ ]:
# STORYBOARD + T2V PROMPTS

all_sec = []
for ch in outline.get('chapters', []):
    for sec in ch.get('sections', []):
        d = dict(**sec); d['chapter_title'] = ch['title']; d['chapter'] = ch.get('chapter', 0)
        all_sec.append(d)

scenes = []; sn = 1; si = 0
for stype in scene_types:
    sec = all_sec[si] if si < len(all_sec) else dict(visual_direction='Wide shot', mood='neutral', chapter_title=TOPIC, section=si+1)
    scenes.append(dict(scene=sn, scene_type=stype,
        title=str(sec.get('chapter_title','')) + ' - Part ' + str(sec.get('section', si+1)),
        visual_direction=sec.get('visual_direction', ''), mood=sec.get('mood', ''),
        duration=SECONDS_PER_SCENE))
    sn += 1; si += 1

ai_sc = [s for s in scenes if s['scene_type'] == 'ai_generated']
total_ai = len(ai_sc); ee = max(1, int(total_ai*0.25)); ls = max(ee+1, int(total_ai*0.75)); last = None
for i, sc in enumerate(ai_sc):
    phase = 'early' if i < ee else ('late' if i >= ls else 'middle')
    pool = arc.get(phase, ['documentary']); avail = [s for s in pool if s != last] or pool
    st = random.choice(avail); sc['assigned_style'] = st; sc['visual_keywords'] = STYLES.get(st, ''); last = st

with open(SESSION_DIR / 'storyboard.json', 'w') as f: json.dump(scenes, f, indent=2)

OPENINGS = ['Start with SHOT TYPE', 'Start with ACTION', 'Start with ENVIRONMENT']
prompts = []
prog = display(HTML(''), display_id=True)
for idx, sc in enumerate(ai_sc):
    st = sc.get('assigned_style', 'documentary'); vk = sc.get('visual_keywords', '')
    prog.update(HTML(progress_html(f'T2V prompt {idx+1}/{len(ai_sc)}', idx, len(ai_sc), 'prompts')))
    sys_p = f'T2V prompt expert. ONE paragraph, 3-5 sentences. Style: {st}. Visual: {vk}. No text overlays.'
    user_p = f'T2V prompt for {SECONDS_PER_SCENE}s:\nVisual: {sc.get("visual_direction","")}\nMood: {sc.get("mood","")}\n{random.choice(OPENINGS)}\nOutput ONLY the prompt text.'
    t2v = query_llm(user_p, sys_p).strip().strip('"')
    prompts.append(dict(scene=sc['scene'], title=sc['title'], t2v_prompt=t2v, style=st, duration=SECONDS_PER_SCENE))

with open(SESSION_DIR / 'prompts.json', 'w') as f: json.dump(prompts, f, indent=2)
prog.update(HTML(progress_html(f'{len(prompts)} prompts done', len(ai_sc), len(ai_sc), 'prompts')))

preview(f'{len(scenes)} scenes in storyboard',
    [f'Scene {s["scene"]:2d} [{s["scene_type"]:12s}] {s.get("assigned_style",""):20s} {s.get("visual_direction","")[:60]}' for s in scenes])
preview(f'{len(prompts)} T2V prompts',
    [f'Scene {p["scene"]} [{p["style"]}]:\n{p["t2v_prompt"]}' for p in prompts])


---
# 2. Media Collection

Archive footage download. Skipped if ARCHIVE_RATIO = 0.


In [ ]:
# ARCHIVE SCRAPER
archive_clips = []
if NUM_ARCHIVE_SCENES > 0 and ARCHIVE_URLS:
    def parse_archive_id(url):
        m = re.search(r'archive\.org/(?:details|download)/([^/?#]+)', url.strip().rstrip('/'))
        if m: return m.group(1)
        raise ValueError('Cannot parse: ' + url)

    MAX_EXTRACT_DURATION = 300  # use first 5 minutes max
    ARCHIVE_DIR = SESSION_DIR / 'archive_clips'; ARCHIVE_DIR.mkdir(exist_ok=True)
    ATEMP = Path('/content/temp_archive'); ATEMP.mkdir(exist_ok=True)

    # How many clips do we need per URL (spread evenly)
    clips_per_url = max(1, NUM_ARCHIVE_SCENES // len(ARCHIVE_URLS))
    extra = NUM_ARCHIVE_SCENES % len(ARCHIVE_URLS)

    for i, url in enumerate(ARCHIVE_URLS):
        try: aid = parse_archive_id(url)
        except ValueError as e: print(f'  FAIL: {e}'); continue
        status(f'Downloading [{i+1}/{len(ARCHIVE_URLS)}] {aid}')
        out_tpl = str(ATEMP / (aid + '.%(ext)s'))
        try: subprocess.run(['yt-dlp', f'https://archive.org/details/{aid}',
            '-f', 'bestvideo[height<=480]+bestaudio/best[height<=480]/best',
            '--merge-output-format', 'mp4', '--no-playlist', '--no-warnings',
            '-o', out_tpl, '--quiet'], check=True, timeout=600)
        except: print('  Download failed'); continue
        video = None
        for fp in ATEMP.iterdir():
            if fp.stem == aid and fp.suffix in ('.mp4','.mkv','.webm'): video = fp; break
        if not video: continue

        total = get_dur(video)
        clip_dir = ARCHIVE_DIR / aid; clip_dir.mkdir(exist_ok=True)
        usable = min(total, MAX_EXTRACT_DURATION)
        need = clips_per_url + (1 if i < extra else 0)

        # Pick random start positions (not chronological)
        if usable > SECONDS_PER_SCENE * need:
            # Enough room for random positions
            positions = sorted(random.sample(
                range(0, int(usable - SECONDS_PER_SCENE)),
                min(need, int(usable // SECONDS_PER_SCENE))))
        else:
            # Not enough room, use sequential
            positions = list(range(0, int(usable), SECONDS_PER_SCENE))[:need]

        status(f'  {total:.0f}s total, extracting {len(positions)} random clips from first {int(usable)}s')
        for j, start_t in enumerate(positions):
            clip_out = clip_dir / f'archive_{len(archive_clips):03d}.mp4'
            r = subprocess.run(['ffmpeg','-y','-ss',str(start_t),'-i',str(video),
                '-t',str(SECONDS_PER_SCENE),'-c:v','libx264','-preset','fast',
                '-crf','23','-an',str(clip_out)],
                capture_output=True, text=True, timeout=120)
            if r.returncode == 0 and clip_out.exists():
                archive_clips.append(dict(path=str(clip_out),
                    duration=round(min(SECONDS_PER_SCENE, usable-start_t), 1), archive_id=aid))
                status(f'    clip {len(archive_clips)}: @{start_t:.0f}s')
        video.unlink(missing_ok=True)

    with open(SESSION_DIR / 'archive_clips.json', 'w') as f:
        json.dump(dict(clips=archive_clips, total=len(archive_clips)), f, indent=2)
    status(f'{len(archive_clips)} archive clips total')
elif NUM_ARCHIVE_SCENES > 0:
    status('[!] Need archive clips but ARCHIVE_URLS is empty!')
else:
    status('No archive footage needed')


---
# 3. Effects + Voice + AI Video


In [ ]:
# EFFECTS
if ENABLE_EFFECTS and prompts:
    for p in prompts:
        if EFFECT_MODE == 'random':
            p['ffmpeg_effects'] = [random.choice(ALL_EFFECTS)]
        else:
            p['ffmpeg_effects'] = [EFFECT_MODE]
        p['effect_intensity'] = 0.7
    with open(SESSION_DIR / 'prompts.json', 'w') as f: json.dump(prompts, f, indent=2)
    preview(f'Effects assigned ({EFFECT_MODE})',
        [f'Scene {p["scene"]}: {p["ffmpeg_effects"][0]} (intensity {p.get("effect_intensity",0.7)})' for p in prompts])
else:
    status('Effects skipped')


In [ ]:
# VOICE
narration_text = outline.get('narration', essay_narration)
narration_file = AUDIO_DIR / 'narration_full.mp3'
audio_file = None

async def gen_tts(text, path, voice, rate='+0%'):
    await edge_tts.Communicate(text, voice, rate=rate).save(str(path))

prog = display(HTML(''), display_id=True)
prog.update(HTML(progress_html(f'TTS: {len(narration_text.split())} words ({VOICE})', 0, 1, '')))
t0 = _time.time()
try:
    await gen_tts(narration_text, narration_file, voice_name)
    audio_duration = get_dur(narration_file)
    audio_file = {'path': str(narration_file), 'duration': audio_duration}
    with open(SESSION_DIR / 'audio_manifest.json', 'w') as f:
        json.dump({'voice': voice_name, 'file': audio_file}, f, indent=2)
    prog.update(HTML(progress_html(f'Done: {audio_duration:.1f}s', 1, 1, '', f'{_time.time()-t0:.1f}s')))
except Exception as e:
    print(f'[!!] TTS failed: {e}')


In [ ]:
# AI VIDEO GENERATION
model_endpoint = 'fal-ai/wan-t2v'

prog = display(HTML(''), display_id=True)
generated_videos = []; t0 = _time.time()

for idx, p in enumerate(prompts):
    vid_path = VIDEOS_DIR / f'scene_{p["scene"]:03d}.mp4'
    prog.update(HTML(progress_html(f'Generating scene {p["scene"]}', idx, len(prompts), 'videos',
        f'{p["t2v_prompt"][:60]}...')))
    try:
        # Submit and poll (more reliable in Colab than subscribe)
        handle = fal_client.submit(model_endpoint,
            arguments=dict(prompt=p['t2v_prompt'], aspect_ratio='16:9',
                           enable_prompt_expansion=False))
        # Poll until done
        import time as _t
        while True:
            st = handle.status()
            if isinstance(st, fal_client.Completed): break
            if isinstance(st, fal_client.InProgress):
                prog.update(HTML(progress_html(f'Generating scene {p["scene"]}', idx, len(prompts), 'videos',
                    f'In progress...')))
            _t.sleep(3)
        result = handle.get()
        url = None
        if isinstance(result, dict):
            v = result.get('video', result.get('output', ''))
            url = v.get('url', '') if isinstance(v, dict) else v
        if url:
            vid_path.write_bytes(requests.get(url, timeout=120).content)
            generated_videos.append(dict(scene=p['scene'], path=str(vid_path), duration=get_dur(vid_path)))
            print(f'  Scene {p["scene"]}: {get_dur(vid_path):.1f}s')
        else: print(f'  Scene {p["scene"]}: no URL in response')
    except Exception as e: print(f'  Scene {p["scene"]} FAIL: {e}')

with open(SESSION_DIR / 'video_manifest.json', 'w') as f: json.dump(generated_videos, f, indent=2)
prog.update(HTML(progress_html(f'{len(generated_videos)} videos done', len(prompts), len(prompts), 'videos', f'{_time.time()-t0:.0f}s')))


---
# 4. Subtitles + Assembly


In [ ]:
# SUBTITLES
sub_file = SESSION_DIR / 'subtitles.ass'
if ENABLE_SUBTITLES and narration_file.exists():
    from faster_whisper import WhisperModel
    status('Loading Whisper...')
    wm = WhisperModel('base', device='cpu', compute_type='int8')
    all_words = []
    segs, info = wm.transcribe(str(narration_file), word_timestamps=True, language='en')
    for seg in segs:
        if seg.words:
            for w in seg.words:
                all_words.append({'word': w.word.strip(), 'start': w.start, 'end': w.end})
    status(f'{len(all_words)} words transcribed')

    sc_f = 720/512; sz = int(52*sc_f); ol = int(3*sc_f); mg = int(30*sc_f)
    ass = f"""[Script Info]
Title: ROTBOTS Subtitles
ScriptType: v4.00+
PlayResX: 1280
PlayResY: 720

[V4+ Styles]
Format: Name,Fontname,Fontsize,PrimaryColour,SecondaryColour,OutlineColour,BackColour,Bold,Italic,Underline,StrikeOut,ScaleX,ScaleY,Spacing,Angle,BorderStyle,Outline,Shadow,Alignment,MarginL,MarginR,MarginV,Encoding
Style: Default,Arial,{sz},&H0000FFFF,&H000000FF,&H00000000,&H80000000,-1,0,0,0,100,100,0,0,1,{ol},1,2,{mg},{mg},{mg},1

[Events]
Format: Layer,Start,End,Style,Name,MarginL,MarginR,MarginV,Effect,Text
"""
    for w in all_words:
        s = w['start']; e = w['end']
        st = f'{int(s//3600)}:{int((s%3600)//60):02d}:{s%60:05.2f}'
        et = f'{int(e//3600)}:{int((e%3600)//60):02d}:{e%60:05.2f}'
        ass += f'Dialogue: 0,{st},{et},Default,,0,0,0,,{w["word"]}\n'

    sub_file.write_text(ass)
    status(f'subtitles.ass: {len(all_words)} words')
else:
    status('Subtitles skipped' if not ENABLE_SUBTITLES else 'No narration audio')


In [ ]:
# ASSEMBLY

storyboard = json.load(open(SESSION_DIR / 'storyboard.json'))
effects_map = {int(p['scene']): p for p in prompts if p.get('ffmpeg_effects')}
essay = json.load(open(SESSION_DIR / 'essay_script.json')) if (SESSION_DIR / 'essay_script.json').exists() else None
if not archive_clips and (SESSION_DIR / 'archive_clips.json').exists():
    archive_clips = json.load(open(SESSION_DIR / 'archive_clips.json')).get('clips', [])
if not upload_clips and (SESSION_DIR / 'upload_clips.json').exists():
    upload_clips = json.load(open(SESSION_DIR / 'upload_clips.json')).get('clips', [])

def build_effect_filter(name, intensity=0.7):
    i = max(0.0, min(1.0, intensity))
    if name == 'film_grain': return f'noise=alls={int(12*i)}:allf=t'
    if name == 'vhs_artifacts': return f'noise=alls={int(30*i)}:allf=t,rgbashift=rh={int(2*i)}:bh={int(-2*i)}'
    if name == 'celluloid_scratches': return f'noise=c0s={int(15*i)}:c0f=t'
    if name == 'sepia_tone':
        inv=1-i; return f'colorchannelmixer={inv+i*0.393:.3f}:{i*0.769:.3f}:{i*0.189:.3f}:0:{i*0.349:.3f}:{inv+i*0.686:.3f}:{i*0.168:.3f}:0:{i*0.272:.3f}:{i*0.534:.3f}:{inv+i*0.131:.3f}:0'
    if name == 'bw_transition':
        inv=1-i; return f'colorchannelmixer={inv+i*0.3:.3f}:{i*0.4:.3f}:{i*0.3:.3f}:0:{i*0.3:.3f}:{inv+i*0.4:.3f}:{i*0.3:.3f}:0:{i*0.3:.3f}:{i*0.4:.3f}:{inv+i*0.3:.3f}:0'
    if name == 'color_grade_warm': return f'eq=saturation={1+0.1*i:.3f}:brightness={0.02*i:.3f}'
    if name == 'color_grade_cool': return f'eq=saturation={1-0.1*i:.3f}'
    if name == 'vignette': return f'vignette=PI/4*{i:.3f}'
    if name == 'flicker': return f'noise=alls={int(8*i)}:allf=t'
    if name == 'desaturate': return f'eq=saturation={0.5+0.5*(1-i):.3f}'
    return None

status(f'Building {len(storyboard)} scenes...')
norm = []; arc_idx = 0; upl_idx = 0
for sc in storyboard:
    sn = sc['scene']; stype = sc['scene_type']
    out = TEMP / f'scene_{sn:03d}.mp4'; src = None
    if stype == 'ai_generated':
        c = VIDEOS_DIR / f'scene_{sn:03d}.mp4'
        if c.exists(): src = c
    elif stype == 'archive':
        if archive_clips:
            src = Path(archive_clips[arc_idx % len(archive_clips)]['path']); arc_idx += 1
    elif stype == 'upload':
        if upload_clips:
            src = Path(upload_clips[upl_idx % len(upload_clips)]['path']); upl_idx += 1
    if not src or not src.exists():
        print(f'  Scene {sn} ({stype}): MISSING'); continue
    vf = 'scale=1280:720:force_original_aspect_ratio=decrease,pad=1280:720:(ow-iw)/2:(oh-ih)/2:black'
    if stype == 'ai_generated' and sn in effects_map:
        p = effects_map[sn]
        for eff in p.get('ffmpeg_effects', []):
            ef = build_effect_filter(eff, p.get('effect_intensity', 0.7))
            if ef: vf += ',' + ef
    if ff(['ffmpeg','-y','-i',str(src),'-vf',vf,'-r','24','-pix_fmt','yuv420p',
           '-c:v','libx264','-preset','fast','-crf','23','-an',
           '-t',str(sc.get('duration',5)),str(out)], f'Scene {sn} ({stype})'):
        norm.append(out)
status(f'{len(norm)}/{len(storyboard)} scenes normalized')

credits_path = None
if ENABLE_CREDITS:
    title = essay.get('title', 'Untitled') if essay else 'Untitled'
    sources_list = essay.get('credits', {}).get('sources', []) if essay else []
    lines = [title, '', 'Generated by ROTBOTS', '', '-- Sources --'] + [s[:60] for s in sources_list]
    lines += ['', '-- Pipeline --', 'LLM: Groq', 'Video: fal.ai', 'Voice: Edge-TTS', 'FFmpeg', '', 'LI-MA TDA 2026']
    D = 8; LH = 42; spd = (720 + len(lines) * LH) / D
    flt = [f"drawtext=text=\'{l.replace(chr(39), chr(8217)).replace(chr(58), chr(92)+chr(58))}\'"
           f':fontsize=28:fontcolor=white:x=(w-text_w)/2:y=h+{i*LH}-{spd:.1f}*t'
           for i, l in enumerate(lines) if l]
    credits_path = TEMP / 'credits.mp4'
    ff(['ffmpeg','-y','-f','lavfi','-i',f'color=c=black:s=1280x720:d={D}:r=24',
        '-vf',','.join(flt),'-pix_fmt','yuv420p','-c:v','libx264','-preset','fast',
        str(credits_path)], 'Credits')

cf = TEMP / 'concat.txt'
with open(cf, 'w') as f:
    for v in norm: f.write(f"file \'{v}\'\n")
    if credits_path and credits_path.exists(): f.write(f"file '{credits_path}'\n")
concat_out = TEMP / 'concatenated.mp4'
ff(['ffmpeg','-y','-f','concat','-safe','0','-i',str(cf),'-c','copy',str(concat_out)], 'Concat')
video_duration = get_dur(concat_out)

audio_out = TEMP / 'with_audio.mp4'
if narration_file.exists():
    narr_padded = TEMP / 'narration_padded.mp3'
    ff(['ffmpeg','-y','-i',str(narration_file),'-af',f'apad=whole_dur={video_duration}',
        '-c:a','libmp3lame','-b:a','128k',str(narr_padded)], 'Pad audio')
    ff(['ffmpeg','-y','-i',str(concat_out),'-i',str(narr_padded),'-c:v','copy',
        '-c:a','aac','-b:a','192k','-map','0:v:0','-map','1:a:0',str(audio_out)], 'Merge audio')
else:
    shutil.copy2(str(concat_out), str(audio_out))

final = SESSION_DIR / 'final_video.mp4'
if ENABLE_SUBTITLES and sub_file.exists():
    local_ass = TEMP / 'subtitles.ass'
    shutil.copy2(str(sub_file), str(local_ass))
    if not ff(['ffmpeg','-y','-i',str(audio_out),'-vf',f'ass={local_ass}',
               '-c:v','libx264','-preset','fast','-crf','23','-c:a','copy',str(final)], 'Burn subtitles'):
        shutil.copy2(str(audio_out), str(final))
else:
    shutil.copy2(str(audio_out), str(final))

status(f'Final: {get_dur(final):.1f}s, {final.stat().st_size/(1024*1024):.1f}MB')


In [ ]:
# WATCH
title = essay.get('title', 'Untitled') if essay else 'Untitled'
if final.exists():
    display(Markdown(f'# {title}\n*Generated by ROTBOTS*\n\n---'))
    try: display(Video(str(final), embed=True, width=720))
    except: print(f'Video saved: {final}')

ai_count = sum(1 for s in storyboard if s['scene_type'] == 'ai_generated')
arc_count = sum(1 for s in storyboard if s['scene_type'] == 'archive')
upl_count = sum(1 for s in storyboard if s['scene_type'] == 'upload')
status(f'Scenes: {ai_count} AI + {arc_count} archive + {upl_count} upload')
status(f'Narration: {len(narration_text.split())} words | Voice: {VOICE}')
status(f'Style: {STYLE_ARC} | Effects: {EFFECT_MODE}')
status(f'Final: {get_dur(final):.1f}s, {final.stat().st_size/(1024*1024):.1f}MB')


---

Every step: automated decisions. What does that mean?

---
*ROTBOTS -- LI-MA TDA 2026, Amsterdam*
